# Anonymization Middleware Demo

This notebook demonstrates the `AnonymizationMiddleware` integrated into a LangChain ReAct agent.

**What it shows:**
1. A user message containing PII (name, email, phone) is **anonymized** before reaching the LLM
2. The LLM sees only fake values and responds using them
3. The LLM response is **deanonymized** — original PII values are restored
4. The anonymization mapping table is printed for inspection

**Key classes used:**
- `AnonymizationMiddleware` — the middleware
- `AnonymizationConfig` — configuration (entity types, Faker locale, fuzzy threshold)
- `PresidioDetectorConfig` — Presidio detector settings (which entity types to detect)

In [10]:
%load_ext autoreload
%autoreload 2

import sys

from dotenv import load_dotenv
from rich import print  # noqa: F401

assert load_dotenv(verbose=True)
sys.path.append(".")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Build the anonymization middleware

In [11]:
from genai_tk.agents.langchain.middleware.anonymization_middleware import (
    AnonymizationConfig,
    AnonymizationMiddleware,
)
from genai_tk.agents.langchain.middleware.presidio_detector import (
    CustomRecognizerConfig,
    PresidioDetectorConfig,
)

# After %autoreload 2 reloads, Pydantic validators can reference stale class objects.
# model_rebuild(force=True) re-compiles the schema against current class references.
AnonymizationConfig.model_rebuild(force=True)

anon_config = AnonymizationConfig(
    detector=PresidioDetectorConfig(
        analyzed_fields=["PERSON", "EMAIL_ADDRESS", "PHONE_NUMBER", "CREDIT_CARD", "LOCATION"],
        enable_spacy=True,
        spacy_model="en_core_web_sm",
        custom_recognizers=[
            CustomRecognizerConfig(
                entity_name="PHONE_NUMBER",
                patterns=[
                    r"\+1[-.\s]?\d{3}[-.\s]?\d{3}[-.\s]?\d{4}",  # +1-555-123-4567 or variants
                    r"\(\d{3}\)[-.\s]?\d{3}[-.\s]?\d{4}",  # (555) 123-4567
                    r"\d{3}[-.\s]?\d{3}[-.\s]?\d{4}",  # 555-123-4567
                ],
                context=["call", "phone", "mobile", "number", "reach"],
                score=0.95,
            )
        ],
    ),
    faker_seed=42,  # reproducible fake values
    faker_locales=["en_US"],
    fuzzy_deanonymize=True,
    fuzzy_threshold=85,
)

middleware = AnonymizationMiddleware(config=anon_config)
print("Middleware created:", middleware)

2026-04-29 20:37:16.678 | INFO     | genai_tk.utils.spacy_model_mngr:setup_spacy_model:103 - SpaCy model 'en_core_web_sm' is available globally


Middleware created: <genai_tk.agents.langchain.middleware.anonymization_middleware.AnonymizationMiddleware object 
at 0x7c7b16a83e90>

## 2. Low-level demo: anonymize / deanonymize directly

In [12]:
DEMO_TEXT = (
    "Hello, my name is John Smith and you can reach me at john.smith@acme.com "
    "or call +1-555-867-5309. I'm based in Paris."
)

THREAD_ID = "demo-thread-1"

anonymized = middleware._anonymize_text(DEMO_TEXT, THREAD_ID)

print("[bold]Original:[/bold]")
print(DEMO_TEXT)
print()
print("[bold]Anonymized (what the LLM sees):[/bold]")
print(anonymized)

2026-04-29 20:37:17.247 | DEBUG    | genai_tk.agents.langchain.middleware.anonymization_middleware:_anonymize_text:256 - [Anonymization] Anonymized 4 entities for thread 'demo-thread-1'


Original:

Hello, my name is John Smith and you can reach me at john.smith@acme.com or call +1-555-867-5309. I'm based in 
Paris.

Anonymized (what the LLM sees):

Hello, my name is Allison Hill and you can reach me at donaldgarcia@example.net or call +1-219-560-0133. I'm based 
in Robinsonshire.

In [13]:
from rich.table import Table

mapping = middleware._mapping.get(THREAD_ID, {})

table = Table(title="PII → Fake Mapping")
table.add_column("Original (PII)", style="red")
table.add_column("Fake (sent to LLM)", style="green")

for original, fake in mapping.items():
    table.add_row(original, fake)

print(table)

                PII → Fake Mapping                
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Original (PII)      ┃ Fake (sent to LLM)       ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ John Smith          │ Allison Hill             │
│ john.smith@acme.com │ donaldgarcia@example.net │
│ +1-555-867-5309     │ +1-219-560-0133          │
│ Paris               │ Robinsonshire            │
└─────────────────────┴──────────────────────────┘

In [14]:
# Simulate an LLM response that uses the fake values
fake_name = None
fake_email = None
for orig, fake in mapping.items():
    if "John Smith" in orig:
        fake_name = fake
    if "john.smith" in orig:
        fake_email = fake

simulated_llm_response = (
    f"I've noted your details. I'll send a confirmation to {fake_email or '<fake-email>'} "
    f"for {fake_name or '<fake-name>'}."
)

print("[bold]Simulated LLM response (contains fake values):[/bold]")
print(simulated_llm_response)

deanonymized = middleware._deanonymize_text(simulated_llm_response, THREAD_ID)

print()
print("[bold]Deanonymized response (original PII restored):[/bold]")
print(deanonymized)

Simulated LLM response (contains fake values):

I've noted your details. I'll send a confirmation to donaldgarcia@example.net for Allison Hill.

Deanonymized response (original PII restored):

I've noted your details. I'll send a confirmation to john.smith@acme.com for John Smith.

## 3. Custom Recognizers

`CustomRecognizerConfig` lets you add domain-specific entity types using regex patterns with optional context words. Context words boost the confidence score when they appear near the match.

In [15]:
from genai_tk.agents.langchain.middleware.anonymization_middleware import (
    AnonymizationConfig,
    AnonymizationMiddleware,
)
from genai_tk.agents.langchain.middleware.presidio_detector import (
    CustomRecognizerConfig,
    PresidioDetectorConfig,
)

# Example 1: Using custom recognizers for domain-specific patterns
# ================================================================
# CustomRecognizerConfig lets you add custom entities using regex patterns
# with optional context words. Context words boost the confidence score
# when they appear near the match.

custom_config_with_patterns = AnonymizationConfig(
    detector=PresidioDetectorConfig(
        analyzed_fields=["PERSON", "EMAIL_ADDRESS"],
        enable_spacy=False,  # fast, no spaCy needed for regex-based recognizers
        custom_recognizers=[
            CustomRecognizerConfig(
                entity_name="EMPLOYEE_ID",
                patterns=[r"\bEMP-[A-Z]\d{4}\b"],
                context=["employee", "staff", "hr"],
                score=0.9,
            ),
        ],
    ),
    faker_seed=99,
)

custom_mw_patterns = AnonymizationMiddleware(config=custom_config_with_patterns)

PATTERN_TEXT = "Please contact John Doe (john.doe@corp.com) His employee ID is EMP-A1234 and he reports to EMP-B5678."

PATTERN_THREAD = "pattern-demo"
anonymized_patterns = custom_mw_patterns._anonymize_text(PATTERN_TEXT, PATTERN_THREAD)

print("[bold]Example 1: Regex-based custom recognizers[/bold]")
print("[bold]Original:[/bold]")
print(PATTERN_TEXT)
print()
print("[bold]Anonymized:[/bold]")
print(anonymized_patterns)

# Example 2: Using the custom_presidio_anonymizer with project_names
# ==================================================================
# For handling enumerated lists (companies, products, projects, etc.),
# use CustomizedPresidioAnonymizer which converts lists to pattern recognizers.

from genai_tk.extra.custom_presidio_anonymizer import CustomizedPresidioAnonymizer

list_anonymizer = CustomizedPresidioAnonymizer(
    company_names=["Acme Corp", "Tech Solutions"],
    product_names=["WidgetPro", "CloudMaster"],
    project_names=["ProjectAlpha", "ProjectBeta"],
    faker_seed=42,
)

PROJECT_TEXT = (
    "John Smith from Acme Corp is working on ProjectAlpha using WidgetPro. "
    "Later he'll move to ProjectBeta at Tech Solutions with CloudMaster."
)

anonymized_lists = list_anonymizer.anonymize(PROJECT_TEXT)

print()
print("[bold]Example 2: List-based recognizers (companies, products, projects)[/bold]")
print("[bold]Original:[/bold]")
print(PROJECT_TEXT)
print()
print("[bold]Anonymized:[/bold]")
print(anonymized_lists)

2026-04-29 20:37:18.834 | DEBUG    | genai_tk.agents.langchain.middleware.anonymization_middleware:_anonymize_text:256 - [Anonymization] Anonymized 4 entities for thread 'pattern-demo'


Example 1: Regex-based custom recognizers

Original:

Please contact John Doe (john.doe@corp.com) His employee ID is EMP-A1234 and he reports to EMP-B5678.

Anonymized:

...

Example 2: List-based recognizers (companies, products, projects)

Original:

John Smith from Acme Corp is working on ProjectAlpha using WidgetPro. Later he'll move to ProjectBeta at Tech 
Solutions with CloudMaster.

Anonymized:

Donald Walker from COMP8196 is working on PROJ0133 using PROD8386. Later he'll move to PROJ7940 at COMP6542 with 
PROD5116.

In [16]:
from rich.table import Table

# Show mapping from Example 1
patterns_mapping = custom_mw_patterns._mapping.get(PATTERN_THREAD, {})

if patterns_mapping:
    table = Table(title="Example 1: Custom Entity Mapping")
    table.add_column("Original", style="red")
    table.add_column("Fake (seen by LLM)", style="green")

    for original, fake in patterns_mapping.items():
        table.add_row(original, fake)

    print(table)

# Show mapping from Example 2
lists_mapping = list_anonymizer.get_mapping()

if lists_mapping:
    table2 = Table(title="Example 2: Project-based Mapping")
    table2.add_column("Original", style="red")
    table2.add_column("Fake (seen by LLM)", style="green")

    for original, fake in lists_mapping.items():
        table2.add_row(original, fake)

    print()
    print(table2)

# Round-trip test: deanonymize
restored = list_anonymizer.deanonymize(anonymized_lists)
print()
print("[bold]Round-trip (deanonymized):[/bold]")
print(restored)
assert restored == PROJECT_TEXT, "Round-trip failed!"
print("[green]✓ Round-trip OK[/green]")

       Example 1: Custom Entity Mapping       
┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Original          ┃ Fake (seen by LLM)     ┃
┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ John Doe          │ Curtis White           │
│ john.doe@corp.com │ deanmiller@example.net │
│ EMP-A1234         │ EMPL6881               │
│ EMP-B5678         │ EMPL9736               │
└───────────────────┴────────────────────────┘

   Example 2: Project-based Mapping    
┏━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┓
┃ Original       ┃ Fake (seen by LLM) ┃
┡━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━┩
│ John Smith     │ Donald Walker      │
│ Acme Corp      │ COMP8196           │
│ ProjectAlpha   │ PROJ0133           │
│ WidgetPro      │ PROD8386           │
│ ProjectBeta    │ PROJ7940           │
│ Tech Solutions │ COMP6542           │
│ CloudMaster    │ PROD5116           │
└────────────────┴────────────────────┘

Round-trip (deanonymized):

John Smith from Acme Corp is working on ProjectAlpha using WidgetPro. Later he'll move to ProjectBeta at Tech 
Solutions with CloudMaster.

✓ Round-trip OK

## 4. Full agent demo with AnonymizationMiddleware

Connect to a real LLM and run a ReAct agent with the middleware attached.

In [17]:
from langchain.agents import create_agent
from langchain_core.tools import tool

from genai_tk.core.llm_factory import get_llm


@tool
def lookup_contact(name: str) -> str:
    """Look up contact information for a person by name."""
    # Fake directory — will receive anonymized names
    return f"Contact for '{name}': Department=Engineering, Extension=4242"


# Reset mapping for a fresh demo thread
AGENT_THREAD = "agent-demo-thread"
middleware.cleanup(AGENT_THREAD)

# Use default LLM (configured in baseline.yaml)
llm = get_llm()

agent = create_agent(
    model=llm,
    tools=[lookup_contact],
    middleware=[middleware],
    system_prompt="You are a helpful assistant. Use the lookup_contact tool to find contact info.",
)

print("Agent ready")

2026-04-29 20:37:20.604 | DEBUG    | genai_tk.core.llm_factory:get_llm:1128 - get LLM:'openai/gpt-oss-120b@openrouter'


Agent ready

In [18]:
from langchain_core.messages import HumanMessage

from genai_tk.agents.langchain.langchain_agent import _extract_content

USER_MESSAGE = "Hi, I'm Alice Johnson (alice.johnson@company.com, +1-555-234-5678). Can you look up my contact record?"

print("[bold cyan]User message (with real PII):[/bold cyan]")
print(USER_MESSAGE)
print()

result = agent.invoke(
    {"messages": [HumanMessage(content=USER_MESSAGE)]},
    config={"configurable": {"thread_id": AGENT_THREAD}},
)


final_response = _extract_content(result)

print("[bold green]Final response (PII restored by deanonymization):[/bold green]")
print(final_response)

User message (with real PII):

Hi, I'm Alice Johnson (alice.johnson@company.com, +1-555-234-5678). Can you look up my contact record?

2026-04-29 20:37:20.674 | DEBUG    | genai_tk.agents.langchain.middleware.anonymization_middleware:_anonymize_text:256 - [Anonymization] Anonymized 3 entities for thread 'agent-demo-thread'


Final response (PII restored by deanonymization):

In [19]:
# Inspect the mapping for this agent thread
agent_mapping = middleware._mapping.get(AGENT_THREAD, {})

table = Table(title=f"Mapping for thread '{AGENT_THREAD}'")
table.add_column("Original (PII)", style="red")
table.add_column("Fake (seen by LLM)", style="green")

for original, fake in agent_mapping.items():
    table.add_row(original, fake)

print(table)

        Mapping for thread 'agent-demo-thread'         
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Original (PII)            ┃ Fake (seen by LLM)      ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Alice Johnson             │ Ryan Munoz              │
│ alice.johnson@company.com │ blairamanda@example.com │
│ +1-555-234-5678           │ 584-695-9310            │
└───────────────────────────┴─────────────────────────┘

## 4. YAML configuration equivalent

To use this middleware in a profile defined in `config/agents/langchain.yaml`:

```yaml
langchain_agents:
  profiles:
    - name: PrivacyAgent
      type: react
      llm: default
      middlewares:
        - class: genai_tk.agents.langchain.middleware.anonymization_middleware:AnonymizationMiddleware
          analyzed_fields:
            - PERSON
            - EMAIL_ADDRESS
            - PHONE_NUMBER
            - CREDIT_CARD
          faker_seed: 42
          fuzzy_deanonymize: true
          fuzzy_threshold: 85
```

Then run with:
```bash
uv run cli agents langchain --profile PrivacyAgent --chat
```